In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

class ModeloNCF(nn.Module):
    def __init__(self, num_usuarios, num_items, dim_latente=8):
        super().__init__()
        self.emb_usuario = nn.Embedding(num_usuarios, dim_latente)
        self.emb_item = nn.Embedding(num_items, dim_latente)
        self.mlp = nn.Sequential(
            nn.Linear(dim_latente * 2, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid() # Salida entre 0 y 1
        )

    def forward(self, id_usuario, id_item):
        vec_u = self.emb_usuario(id_usuario)
        vec_i = self.emb_item(id_item)
        concatenado = torch.cat([vec_u, vec_i], dim=1)
        return self.mlp(concatenado).squeeze() # Squeeze quita la dimensión extra

class MovieLensDataset(Dataset):
    def __init__(self, ruta_archivo):
        # MovieLens usa tabuladores para separar: user_id | item_id | rating | timestamp
        df = pd.read_csv(ruta_archivo, sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])
        
        self.users = torch.tensor(df['user_id'].values, dtype=torch.long)
        self.items = torch.tensor(df['item_id'].values, dtype=torch.long)
        
        # Escalar ratings de [1, 5] a [0, 1] para que coincida con la salida Sigmoid
        ratings_escalados = (df['rating'].values - 1) / 4.0
        self.ratings = torch.tensor(ratings_escalados, dtype=torch.float32)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.ratings[idx]

In [ ]:
# Cargar Datasets directamente de la URL
train_dataset = MovieLensDataset('https://files.grouplens.org/datasets/movielens/ml-100k/u1.base')
test_dataset = MovieLensDataset('https://files.grouplens.org/datasets/movielens/ml-100k/u1.test')

# DataLoaders: Agrupan los datos en "batches" (lotes) para usar Descenso de Gradiente Estocástico (SGD/Adam)
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

# Determinar dimensiones máximas (para no salirnos del índice de la tabla de Embedding)
num_usuarios = max(train_dataset.users.max(), test_dataset.users.max()) + 1
num_items = max(train_dataset.items.max(), test_dataset.items.max()) + 1

# Inicializar Modelo, Función de Pérdida y Optimizador
model = ModeloNCF(num_usuarios, num_items, dim_latente=16)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

In [ ]:
epochs = 20

print("Iniciando entrenamiento...")
for epoch in range(epochs):
    
    # --- FASE DE ENTRENAMIENTO ---
    model.train() # Ponemos la red en modo entrenamiento
    train_loss = 0.0
    
    for usuarios, items, ratings in train_loader:
        optimizer.zero_grad()
        predicciones = model(usuarios, items)
        loss = criterion(predicciones, ratings)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    media_train_loss = train_loss / len(train_loader)
    
    # --- FASE DE EVALUACIÓN ---
    model.eval() # Ponemos la red en modo evaluación (desactiva Dropouts, etc.)
    test_loss = 0.0
    
    with torch.no_grad(): # Apagamos el cálculo de gradientes para ahorrar memoria
        for usuarios, items, ratings in test_loader:
            predicciones = model(usuarios, items)
            loss = criterion(predicciones, ratings)
            test_loss += loss.item()
            
    media_test_loss = test_loss / len(test_loader)
    
    # Para que sea intuitivo, calculamos el RMSE deshaciendo la escala (x4)
    rmse_test_real = np.sqrt(media_test_loss) * 4
    
    print(f"Epoch {epoch+1}/{epochs} | Loss Train: {media_train_loss:.4f} | Loss Test: {media_test_loss:.4f} | RMSE Test Real (1-5): {rmse_test_real:.4f} estrellas")

print("¡Entrenamiento completado!")